In [12]:
import sys
from pathlib import Path

# go from /notebooks → project root
project_root = Path().resolve().parent
sys.path.append(str(project_root))

print(project_root)

D:\Big Boi Files\projects\langchain_practice\Microplastics-Research-Assistant-RAG-System-


In [4]:
# temporary bug workaround. see link for more permament solution. 
# https://github.com/vibrantlabsai/ragas/issues/2753#issuecomment-4563590504
import types
dummy_chat = types.ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = type("ChatVertexAI", (object,), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

import langchain_community.llms
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

In [ ]:
from rag.pipeline import run_rag
from rag.retriever import get_retriever
from rag.ingest import *

# load PDFs
docs = load_documents()

# chunk PDFs
split_docs = split_documents(docs)

# setup system
vector_store = build_vectorstore(split_docs)
retriever = get_retriever(vector_store)

# run experiments
samples = [
    run_rag("What are microplastics doing to human health?", retriever),
    run_rag("How do microplastics enter the ocean?", retriever),
]

In [ ]:
from pprint import pprint
pprint(samples)

[{'answer': 'Microplastics may cause adverse effects such as genotoxicity and '
            'cytotoxicity, as observed in in-vitro studies with human '
            'peripheral blood lymphocytes. These effects have been linked to '
            'disorders like infertility, diabetes, obesity, and cardiovascular '
            'disease. However, the overall health effects in humans are less '
            'well-studied compared to marine organisms, and current evidence '
            'does not suggest a direct concern regarding the colors of '
            'microplastics or their toxicity in drinking-water.',
  'contexts': ['can cause plastics to degrade and to discolour [5]. Overall, '
               'while the dominant colours of microplastics \n'
               'found in humans may vary, there is currently no evidence to '
               'suggest that the colours of microplastics have \n'
               'any direct effects on human health.\n'
               'Size of microplastics in human s

In [17]:
import json

with open("ragas_samples.json", "w", encoding="utf-8") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)

In [15]:
from src.eval import build_ragas_dataset
dataset = build_ragas_dataset(samples)

In [8]:
print(dataset)

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=2)


### Below is experimental and needs modularized

In [7]:
# examples/ragas_examples/improve_rag/evals.py
from ragas.metrics import DiscreteMetric

# Define correctness metric
correctness_metric = DiscreteMetric(
    name="correctness",
    prompt="""Compare the model response to the expected answer and determine if it's correct.

Consider the response correct if it:
1. Contains the key information from the expected answer
2. Is factually accurate based on the provided context
3. Adequately addresses the question asked

Return 'pass' if the response is correct, 'fail' if it's incorrect.

Question: {question}
Expected Answer: {expected_answer}
Model Response: {response}

Evaluation:""",
    allowed_values=["pass", "fail"],
)

In [9]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, 
    answer_relevancy,
    context_precision,
    context_recall)

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    is_async=False
)

print(result)

C:\Users\dorky\AppData\Local\Temp\ipykernel_24492\3380665120.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\dorky\AppData\Local\Temp\ipykernel_24492\3380665120.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\dorky\AppData\Local\Temp\ipykernel_24492\3380665120.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\dorky\AppData\Local\Temp\ipykernel_24

TypeError: evaluate() got an unexpected keyword argument 'is_async'